In [ ]:
import sys
from pathlib import Path

project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

print("Project root:", project_root)

In [ ]:
import rasterio

dem_path = "../data/raw/kamrup_metro_dem.tif"

with rasterio.open(dem_path) as src:
    bounds = src.bounds
    pixel_width_deg, pixel_height_deg = src.res

    print("Shape:", src.shape)
    print("CRS:", src.crs)
    print("Resolution:", src.res)
    print("Bounds:", bounds)

In [ ]:
import numpy as np
import rasterio

with rasterio.open(dem_path) as src:
    dem = src.read(1)
    nodata = src.nodata

dem = dem.astype(float)
dem[dem == nodata] = np.nan

print("NoData value:", nodata)
print("Elevation array shape:", dem.shape)
print("Minimum elevation:", np.nanmin(dem), "m")
print("Maximum elevation:", np.nanmax(dem), "m")
print("Mean elevation:", np.nanmean(dem), "m")

In [ ]:
from app.terrain_utils import compute_slope

# Convert the DEM's geographic resolution to metres
center_lat = (bounds.top + bounds.bottom) / 2

meters_per_degree_lat = 111320
meters_per_degree_lon = 111320 * np.cos(np.radians(center_lat))

cellsize_y = pixel_height_deg * meters_per_degree_lat
cellsize_x = pixel_width_deg * meters_per_degree_lon

# Compute slope using the reusable function
slope = compute_slope(
    dem,
    (cellsize_y, cellsize_x)
)

print("Slope shape:", slope.shape)
print("Minimum slope:", np.nanmin(slope), "degrees")
print("Maximum slope:", np.nanmax(slope), "degrees")
print("Mean slope:", np.nanmean(slope), "degrees")

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Elevation
elevation_plot = axes[0].imshow(dem, cmap="terrain")
axes[0].set_title("Kamrup Metropolitan - Elevation")
axes[0].set_xlabel("Column")
axes[0].set_ylabel("Row")
fig.colorbar(elevation_plot, ax=axes[0], label="Elevation (m)")

# Slope
slope_plot = axes[1].imshow(slope, cmap="viridis")
axes[1].set_title("Kamrup Metropolitan - Slope")
axes[1].set_xlabel("Column")
axes[1].set_ylabel("Row")
fig.colorbar(slope_plot, ax=axes[1], label="Slope (degrees)")

plt.tight_layout()
plt.show()